In [1]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np
from matplotlib import image
from matplotlib import pyplot as plt
import os
from tensorflow import keras
from PIL import Image

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
import cv2

In [ ]:
master_dir = "v_2"
maps = ["urban","agri","barrenland","grassland"]
for i in os.listdir(master_dir):
    print(i)

In [ ]:
x = []
y = []
img_size = 128
for i in os.listdir(master_dir):
    if i==".DS_Store":
        continue
    subfolder = master_dir+"/"+i
    input_folder = subfolder+"/"+"s1"
    output_folder = subfolder+"/"+'s2'
    print(input_folder)
    print(output_folder)
    for file in os.listdir(output_folder)[:1000]:
        if file.endswith(('.png','.jpeg','.jpg')):
            img_path = os.path.join(output_folder, file)
            img = Image.open(img_path).resize( ( img_size , img_size ))
            img_array = np.array(img, dtype=np.float64)
            img_array/=255
            y.append(img_array)
    for file in os.listdir(input_folder)[:1000]:
        if file.endswith(('.png','.jpeg','.jpg')):
            img_path = os.path.join(input_folder, file)
            img = Image.open(img_path).convert('L').resize( ( img_size , img_size ))
            img_array = np.array(img, dtype=np.float64)
            img_array = img_array.reshape((img_size, img_size, 1))
            img_array/=255
            x.append(img_array)

In [ ]:
x = np.array(x)
y = np.array(y)

In [ ]:
x.shape, y.shape

In [1]:
x_train, x_test, y_train,y_test = train_test_split(x,y,test_size=0.15,random_state=56, shuffle=True)

NameError: name 'train_test_split' is not defined

In [ ]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
from keras.layers import Conv2D, LeakyReLU, Concatenate, Conv2DTranspose, Input
from keras.models import Model
from keras.optimizers import Adam

img_size = 128

In [ ]:
def get_generator_model():
    inputs = Input(shape=(img_size, img_size, 1))

    # Reduced number of filters and layers
    conv1 = Conv2D(8, kernel_size=(3, 3), strides=1)(inputs)
    conv1 = LeakyReLU()(conv1)
    conv1 = Conv2D(16, kernel_size=(3, 3), strides=1)(conv1)
    conv1 = LeakyReLU()(conv1)

    bottleneck = Conv2D(16, kernel_size=(3, 3), strides=1, activation='tanh', padding='same')(conv1)

    concat_1 = Concatenate()([bottleneck, conv1])
    conv_up_1 = Conv2DTranspose(16, kernel_size=(3, 3), strides=1, activation='relu')(concat_1)
    conv_up_1 = Conv2DTranspose(8, kernel_size=(3, 3), strides=1, activation='relu')(conv_up_1)
    conv_up_1 = Conv2DTranspose(3, kernel_size=(3, 3), strides=1, activation='relu')(conv_up_1)

    model = Model(inputs, conv_up_1)
    return model

In [ ]:
generator = get_generator_model()
generator.compile(optimizer=Adam(learning_rate=0.005), loss='mean_squared_error')

generator.summary()

In [ ]:
from keras.layers import MaxPooling2D, Flatten, Dense
from keras.models import Sequential

def create_discriminator():
    model = Sequential([
        # Reduced number of Conv2D layers and filters
        Conv2D(16, kernel_size=(3,3), strides=1, input_shape=(128, 128, 3), activation='relu'),
        MaxPooling2D(),
        Conv2D(32, kernel_size=(3,3), strides=1, activation='relu'),
        MaxPooling2D(),
        Flatten(),
        Dense(32, activation='relu'),  # Reduced neurons
        Dense(1, activation='sigmoid')
    ])
    return model

discriminator = create_discriminator()
discriminator.compile(optimizer=Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])

discriminator.summary()

In [ ]:
discriminator.compile(optimizer=Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from keras.losses import BinaryCrossentropy, MeanSquaredError
cross_entropy = BinaryCrossentropy()
mse = MeanSquaredError()

def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output) - 0.1 * tf.random.uniform(tf.shape(real_output))
    fake_labels = 0.1 * tf.random.uniform(tf.shape(fake_output))
    real_loss = cross_entropy(real_labels, real_output)
    fake_loss = cross_entropy(fake_labels, fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output, real_y):
    loss = mse(fake_output, tf.cast(real_y, tf.float32))
    return loss

In [ ]:
generator_optimizer = Adam(0.0005)
discriminator_optimizer = Adam(0.0005)

@tf.function
def train_step(x_input, y_actual):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        y_gen = generator(x_input, training=True)
        real_output = discriminator(y_actual, training=True)
        generated_output = discriminator(y_gen, training=True)
        gen_loss = generator_loss(y_gen, y_actual)
        disc_loss = discriminator_loss(real_output, generated_output)
    
    generator_gradient = gen_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradient = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    generator_optimizer.apply_gradients(zip(generator_gradient, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradient, discriminator.trainable_variables))
    return gen_loss, disc_loss

epochs = 1000
batch_size = 32  # Reduced batch size
num_batches = x_train.shape[0] // batch_size

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    for batch in range(num_batches):
        start = batch * batch_size
        end = (batch + 1) * batch_size
        x_batch = x_train[start:end]
        y_batch = y_train[start:end]
        gen_loss, disc_loss = train_step(x_batch, y_batch)
        print(f"Batch {batch+1}/{num_batches} - Generator Loss: {gen_loss.numpy()}, Discriminator Loss: {disc_loss.numpy()}")
    
    if (epoch + 1) % 10 == 0:
        print(f"Saving weights at epoch {epoch + 1}")
        generator.save_weights('generator.weights.h5')
        discriminator.save_weights('discriminator.weights.h5')